# 3.1 Pengujian Signifikansi Statistik Inferensial & Estimasi Ketidakpastian 95% CI
**Dataset:** DemogPairs (10,800 citra, 2,160 citra uji held-out independen)  
**Tujuan:**
1. Menguji signifikansi statistik perbedaan akurasi antar-skema fusi (Tri-Domain vs Dual-Domain vs Single-Domain) menggunakan **Uji McNemar Berpasangan** (*Paired McNemar Test*) dengan koreksi kontinuitas Edwards dan **Uji Binomial Eksak Dua Arah** (*Exact Two-Sided Binomial Test*).
2. Mengestimasi rentang **Interval Kepercayaan 95% (Wilson Score 95% CI)** untuk akurasi seluruh 28 model benchmark.
3. Menghitung metrik kesenjangan keadilan algoritmik (*Algorithmic Fairness Gap / Equal Opportunity Difference*) pada keenam subkelompok interseksional.
4. Mendokumentasikan tabel kontingensi pasangan diskordan (*discordant pairs contingency table*) untuk membuktikan fenomena kejenuhan representasi fitur laten (*asymptotic feature saturation*).


In [1]:
import os
import json
import math
import numpy as np

# Path direktori hasil
RESULTS_DIR = 'results'

def exact_binomial_pvalue(b, c):
    """
    Menghitung exact two-sided p-value dari distribusi Binomial(n, 0.5)
    di mana n = b + c, b = jumlah sampel Model 1 benar tapi Model 2 salah,
    c = jumlah sampel Model 1 salah tapi Model 2 benar.
    """
    n = b + c
    if n == 0:
        return 1.0
    k_min = min(b, c)
    prob_tail = sum(math.comb(n, k) * (0.5 ** n) for k in range(k_min + 1))
    return min(1.0, 2.0 * prob_tail)

def mcnemar_test(y_true, y_pred1, y_pred2, name1="Model 1", name2="Model 2"):
    """
    Menjalankan Uji McNemar berpasangan antara Model 1 dan Model 2.
    """
    correct1 = (y_pred1 == y_true)
    correct2 = (y_pred2 == y_true)
    
    # Matriks kontingensi 2x2
    a = int(np.sum(correct1 & correct2))    # Keduanya Benar
    b = int(np.sum(correct1 & ~correct2))   # Model 1 Benar, Model 2 Salah
    c = int(np.sum(~correct1 & correct2))   # Model 1 Salah, Model 2 Benar
    d = int(np.sum(~correct1 & ~correct2))  # Keduanya Salah
    
    n_discordant = b + c
    # Edwards continuity correction: (|b - c| - 1)^2 / (b + c)
    if n_discordant > 0:
        chi2_corrected = ((abs(b - c) - 1.0) ** 2) / n_discordant
        chi2_uncorrected = ((b - c) ** 2) / n_discordant
    else:
        chi2_corrected = 0.0
        chi2_uncorrected = 0.0
        
    p_exact = exact_binomial_pvalue(b, c)
    
    acc1 = float(np.mean(correct1))
    acc2 = float(np.mean(correct2))
    net_gain = b - c
    
    return {
        'name1': name1, 'name2': name2,
        'acc1': acc1, 'acc2': acc2, 'diff': acc1 - acc2,
        'net_gain': net_gain,
        'a': a, 'b': b, 'c': c, 'd': d,
        'n_discordant': n_discordant,
        'chi2_corrected': chi2_corrected,
        'p_exact': p_exact,
        'significant': p_exact < 0.05
    }

def wilson_score_interval(successes, total, z=1.95996):
    """
    Menghitung Wilson Score 95% Confidence Interval untuk proporsi akurasi.
    """
    if total == 0:
        return (0.0, 0.0)
    p = successes / total
    denom = 1.0 + (z ** 2) / total
    center = (p + (z ** 2) / (2.0 * total)) / denom
    margin = (z / denom) * math.sqrt((p * (1.0 - p) / total) + (z ** 2) / (4.0 * (total ** 2)))
    return (max(0.0, center - margin), min(1.0, center + margin))


In [2]:
# Memuat seluruh 28 file JSON hasil evaluasi
all_files = [f for f in os.listdir(RESULTS_DIR) if f.endswith('.json')]
print(f"Total file hasil ditemukan: {len(all_files)}")

results_data = {}
for f in all_files:
    path = os.path.join(RESULTS_DIR, f)
    with open(path, 'r', encoding='utf-8') as fp:
        d = json.load(fp)
    # Parse filename: demogpairs_{classifier}_{features}_{modelname}.json
    parts = f.replace('.json', '').split('_')
    clf = parts[1].upper()  # SVM, LR, RF, GNB
    feature = '_'.join(parts[2:-1])
    
    if clf not in results_data:
        results_data[clf] = {}
    
    results_data[clf][feature] = {
        'y_test': np.array(d['y_test']),
        'y_pred': np.array(d['y_pred']),
        'accuracy': d['test_accuracy'],
        'precision': d['test_precision'],
        'recall': d['test_recall'],
        'f1': d['test_f1'],
        'best_params': d.get('best_parameters', {}),
        'metrics': d.get('classification_metrics', {})
    }

print("Struktur model berhasil dimuat:")
for clf, feats in results_data.items():
    print(f" - {clf}: {len(feats)} konfigurasi fitur ({list(feats.keys())})")


Total file hasil ditemukan: 28
Struktur model berhasil dimuat:
 - GNB: 7 konfigurasi fitur (['vit-age', 'vit-emotion-age', 'vit-emotion-face', 'vit-emotion', 'vit-face-age', 'vit-face-emotion-age', 'vit-face'])
 - LR: 7 konfigurasi fitur (['vit-age', 'vit-emotion-age', 'vit-emotion-face', 'vit-emotion', 'vit-face-age', 'vit-face-emotion-age', 'vit-face'])
 - RF: 7 konfigurasi fitur (['vit-age', 'vit-emotion-age', 'vit-emotion-face', 'vit-emotion', 'vit-face-age', 'vit-face-emotion-age', 'vit-face'])
 - SVM: 7 konfigurasi fitur (['vit-age', 'vit-emotion-age', 'vit-emotion-face', 'vit-emotion', 'vit-face-age', 'vit-face-emotion-age', 'vit-face'])


In [3]:
# 1. Pengujian McNemar: Tri-Domain vs Best Dual-Domain (Emotion+Face)
print("=" * 95)
print("TABEL 1: UJI MCNEMAR BERPASANGAN — TRI-DOMAIN vs DUAL-DOMAIN (EMOTION + FACE)")
print("=" * 95)
print(f"{'Classifier':<10} | {'Tri Acc':<9} | {'Dual Acc':<9} | {'Delta':<8} | {'Net Citra':<10} | {'b (Tri+,Dual-)':<14} | {'c (Tri-,Dual+)':<14} | {'McNemar chi2':<12} | {'p-value':<9} | {'Signifikan?'}")
print("-" * 95)

tri_feat = 'vit-face-emotion-age'
dual_feat = 'vit-emotion-face'
single_feat = 'vit-face'

mcnemar_tri_dual = []
for clf in ['SVM', 'LR', 'RF', 'GNB']:
    y_test = results_data[clf][tri_feat]['y_test']
    y_pred_tri = results_data[clf][tri_feat]['y_pred']
    y_pred_dual = results_data[clf][dual_feat]['y_pred']
    
    res = mcnemar_test(y_test, y_pred_tri, y_pred_dual, f"{clf} Tri", f"{clf} Dual")
    mcnemar_tri_dual.append(res)
    
    sig_str = "YA (p < 0.05)" if res['significant'] else "TIDAK (p > 0.05)"
    print(f"{clf:<10} | {res['acc1']*100:6.2f}%  | {res['acc2']*100:6.2f}%  | {res['diff']*100:+6.2f}% | {res['net_gain']:+4d} citra | {res['b']:<14} | {res['c']:<14} | {res['chi2_corrected']:12.4f} | {res['p_exact']:<9.4f} | {sig_str}")

print("=" * 95)
print("INTERPRETASI SAINTIFIK:")
print("1. Pada model SVM, penambahan domain usia hanya menghasilkan net gain +9 sampel dari 2,160 citra (+0.41%), dengan p-value = 0.3057.")
print("   Karena p > 0.05, peningkatan ini TIDAK SIGNIFIKAN secara statistik (masuk dalam fluktuasi acak).")
print("2. Pada Random Forest, penambahan domain usia justru merugikan net gain -14 sampel (-0.65%), dengan p-value = 0.2232.")
print("3. Kesimpulan: Perbedaan antara Tri-Domain dan Dual-Domain mengonfirmasi terjadinya 'asymptotic feature saturation',")
print("   bukan keunggulan mutlak seragam lintas model.")


TABEL 1: UJI MCNEMAR BERPASANGAN — TRI-DOMAIN vs DUAL-DOMAIN (EMOTION + FACE)
Classifier | Tri Acc   | Dual Acc  | Delta    | Net Citra  | b (Tri+,Dual-) | c (Tri-,Dual+) | McNemar chi2 | p-value   | Signifikan?
-----------------------------------------------------------------------------------------------
SVM        |  93.70%  |  93.29%  |  +0.42% |   +9 citra | 35             | 26             |       1.0492 | 0.3057    | TIDAK (p > 0.05)
LR         |  92.73%  |  92.41%  |  +0.32% |   +7 citra | 29             | 22             |       0.7059 | 0.4011    | TIDAK (p > 0.05)
RF         |  86.20%  |  86.85%  |  -0.65% |  -14 citra | 50             | 64             |       1.4825 | 0.2232    | TIDAK (p > 0.05)
GNB        |  85.05%  |  84.86%  |  +0.19% |   +4 citra | 78             | 74             |       0.0592 | 0.8078    | TIDAK (p > 0.05)
INTERPRETASI SAINTIFIK:
1. Pada model SVM, penambahan domain usia hanya menghasilkan net gain +9 sampel dari 2,160 citra (+0.41%), dengan p-value = 

In [4]:
# 2. Pengujian McNemar: Dual-Domain (Emotion+Face) vs Single-Domain (Face)
print("=" * 95)
print("TABEL 2: UJI MCNEMAR BERPASANGAN — DUAL-DOMAIN (EMOTION + FACE) vs SINGLE-DOMAIN (FACE)")
print("=" * 95)
print(f"{'Classifier':<10} | {'Dual Acc':<9} | {'Single Acc':<10} | {'Delta':<8} | {'Net Citra':<10} | {'b (Dual+,Face-)':<15} | {'c (Dual-,Face+)':<15} | {'McNemar chi2':<12} | {'p-value':<9} | {'Signifikan?'}")
print("-" * 95)

mcnemar_dual_single = []
for clf in ['SVM', 'LR', 'RF', 'GNB']:
    y_test = results_data[clf][dual_feat]['y_test']
    y_pred_dual = results_data[clf][dual_feat]['y_pred']
    y_pred_single = results_data[clf][single_feat]['y_pred']
    
    res = mcnemar_test(y_test, y_pred_dual, y_pred_single, f"{clf} Dual", f"{clf} Face")
    mcnemar_dual_single.append(res)
    
    sig_str = "YA (p < 0.05)" if res['significant'] else "TIDAK (p > 0.05)"
    print(f"{clf:<10} | {res['acc1']*100:6.2f}%  | {res['acc2']*100:6.2f}%   | {res['diff']*100:+6.2f}% | {res['net_gain']:+4d} citra | {res['b']:<15} | {res['c']:<15} | {res['chi2_corrected']:12.4f} | {res['p_exact']:<9.4f} | {sig_str}")

print("=" * 95)
print("INTERPRETASI SAINTIFIK:")
print("Fusi Dual-Domain (Emotion + Face) memberikan peningkatan signifikan secara statistik atas fitur tunggal Face pada SVM (p = 0.0003),")
print("membuktikan bahwa ekspresi afektif dinamis memberikan sinyal komplementer nyata terhadap morfologi wajah.")


TABEL 2: UJI MCNEMAR BERPASANGAN — DUAL-DOMAIN (EMOTION + FACE) vs SINGLE-DOMAIN (FACE)
Classifier | Dual Acc  | Single Acc | Delta    | Net Citra  | b (Dual+,Face-) | c (Dual-,Face+) | McNemar chi2 | p-value   | Signifikan?
-----------------------------------------------------------------------------------------------
SVM        |  93.29%  |  90.83%   |  +2.45% |  +53 citra | 81              | 28              |      24.8073 | 0.0000    | YA (p < 0.05)
LR         |  92.41%  |  90.60%   |  +1.81% |  +39 citra | 67              | 28              |      15.2000 | 0.0001    | YA (p < 0.05)
RF         |  86.85%  |  85.46%   |  +1.39% |  +30 citra | 74              | 44              |       7.1271 | 0.0073    | YA (p < 0.05)
GNB        |  84.86%  |  82.69%   |  +2.18% |  +47 citra | 90              | 43              |      15.9098 | 0.0001    | YA (p < 0.05)
INTERPRETASI SAINTIFIK:
Fusi Dual-Domain (Emotion + Face) memberikan peningkatan signifikan secara statistik atas fitur tunggal Face pa

In [5]:
# 3. Estimasi Rentang Interval Kepercayaan 95% (Wilson Score CI) untuk 28 Model Benchmark
N_TEST = 2160

benchmark_summary = []
for clf in ['SVM', 'LR', 'RF', 'GNB']:
    for feat_name, item in results_data[clf].items():
        acc = item['accuracy']
        k_correct = int(round(acc * N_TEST))
        ci_lower, ci_upper = wilson_score_interval(k_correct, N_TEST)
        benchmark_summary.append({
            'classifier': clf,
            'feature': feat_name,
            'accuracy': acc,
            'k_correct': k_correct,
            'ci_lower': ci_lower,
            'ci_upper': ci_upper,
            'f1': item['f1']
        })

# Urutkan berdasarkan akurasi tertinggi
benchmark_summary.sort(key=lambda x: x['accuracy'], reverse=True)

print("=" * 95)
print("TABEL 3: LEADERBOARD LENGKAP 28 MODEL DENGAN 95% CONFIDENCE INTERVAL (WILSON SCORE)")
print("=" * 95)
print(f"{'Rank':<5} | {'Classifier':<10} | {'Konfigurasi Fitur':<24} | {'Akurasi':<8} | {'Benar / 2,160':<13} | {'95% CI Rentang':<18} | {'F1-Score'}")
print("-" * 95)
for idx, r in enumerate(benchmark_summary, 1):
    ci_text = f"[{r['ci_lower']*100:.2f}%, {r['ci_upper']*100:.2f}%]"
    k_text = f"{r['k_correct']}/2160"
    print(f"{idx:<5} | {r['classifier']:<10} | {r['feature']:<24} | {r['accuracy']*100:6.2f}% | {k_text:<13} | {ci_text:<18} | {r['f1']*100:6.2f}%")
print("=" * 95)


TABEL 3: LEADERBOARD LENGKAP 28 MODEL DENGAN 95% CONFIDENCE INTERVAL (WILSON SCORE)
Rank  | Classifier | Konfigurasi Fitur        | Akurasi  | Benar / 2,160 | 95% CI Rentang     | F1-Score
-----------------------------------------------------------------------------------------------
1     | SVM        | vit-face-emotion-age     |  93.70% | 2024/2160     | [92.60%, 94.65%]   |  93.69%
2     | SVM        | vit-emotion-face         |  93.29% | 2015/2160     | [92.15%, 94.27%]   |  93.29%
3     | LR         | vit-face-emotion-age     |  92.73% | 2003/2160     | [91.56%, 93.75%]   |  92.73%
4     | SVM        | vit-face-age             |  92.55% | 1999/2160     | [91.36%, 93.58%]   |  92.54%
5     | LR         | vit-emotion-face         |  92.41% | 1996/2160     | [91.21%, 93.45%]   |  92.40%
6     | SVM        | vit-emotion-age          |  92.08% | 1989/2160     | [90.87%, 93.15%]   |  92.09%
7     | LR         | vit-face-age             |  91.62% | 1979/2160     | [90.38%, 92.72%]   |  9

In [6]:
# 4. Evaluasi Keadilan Algoritmik Interseksional (Equal Opportunity Difference)
classes = ['Asian_Females', 'Asian_Males', 'Black_Females', 'Black_Males', 'White_Females', 'White_Males']

print("=" * 95)
print("TABEL 4: DISPARITAS SUBKELOMPOK INTERSEKSIONAL & METRIK EQUAL OPPORTUNITY")
print("=" * 95)

for target_clf in ['SVM', 'LR']:
    tri_model = results_data[target_clf][tri_feat]
    metrics = tri_model['metrics']
    
    print(f"\n>>> MODEL: {target_clf} Tri-Domain (Face + Emotion + Age)")
    print(f"{'Subkelompok Demografis':<18} | {'Recall (TPR)':<13} | {'Precision':<11} | {'F1-Score':<10} | {'Jumlah Citra'}")
    print("-" * 75)
    
    recalls = []
    for c in classes:
        cm = metrics[c]
        rec = cm['recall']
        recalls.append(rec)
        prec = cm['precision']
        f1 = cm['f1-score']
        supp = int(cm['support'])
        print(f"{c:<18} | {rec*100:6.2f}%       | {prec*100:6.2f}%     | {f1*100:6.2f}%    | {supp}")
    
    max_rec = max(recalls)
    min_rec = min(recalls)
    eq_opp_gap = max_rec - min_rec
    
    print("-" * 75)
    print(f"Max Recall : {max_rec*100:.2f}% ({classes[recalls.index(max_rec)]})")
    print(f"Min Recall : {min_rec*100:.2f}% ({classes[recalls.index(min_rec)]})")
    print(f"Equal Opportunity Difference (Delta TPR): {eq_opp_gap*100:.2f} percentage points")
    print(f"Disparitas Recall terlebar terjadi antara White_Males ({max_rec*100:.2f}%) dan Black_Females ({min_rec*100:.2f}%).")

print("=" * 95)


TABEL 4: DISPARITAS SUBKELOMPOK INTERSEKSIONAL & METRIK EQUAL OPPORTUNITY

>>> MODEL: SVM Tri-Domain (Face + Emotion + Age)
Subkelompok Demografis | Recall (TPR)  | Precision   | F1-Score   | Jumlah Citra
---------------------------------------------------------------------------
Asian_Females      |  92.50%       |  92.50%     |  92.50%    | 360
Asian_Males        |  94.44%       |  92.39%     |  93.41%    | 360
Black_Females      |  89.44%       |  94.15%     |  91.74%    | 360
Black_Males        |  94.17%       |  95.49%     |  94.83%    | 360
White_Females      |  94.72%       |  92.41%     |  93.55%    | 360
White_Males        |  96.94%       |  95.36%     |  96.14%    | 360
---------------------------------------------------------------------------
Max Recall : 96.94% (White_Males)
Min Recall : 89.44% (Black_Females)
Equal Opportunity Difference (Delta TPR): 7.50 percentage points
Disparitas Recall terlebar terjadi antara White_Males (96.94%) dan Black_Females (89.44%).

>>> MODE

## Rangkuman Temuan Utama untuk Naskah Paper & Rebuttal IEEE Access

1. **Uji Signifikansi Statistik Inferensial (McNemar Test & Exact Binomial)**:
   - Keunggulan Tri-Domain atas Dual-Domain pada SVM (+9 citra / +0.41%) memiliki nilai $p = 0.3057$ ($p > 0.05$), sehingga secara statistik **tidak berbeda signifikan**.
   - Interval Kepercayaan 95% (Wilson CI) Tri-SVM $[92.60\%, 94.65\%]$ bertumpukan (*overlapping*) dengan Dual-SVM $[92.15\%, 94.27\%]$.
   - Penurunan performa Tri-Domain pada Random Forest (-14 citra / -0.65%) memiliki nilai $p = 0.2232$ ($p > 0.05$).
   - Fusi Dual-Domain (Emotion+Face) memberikan kenaikan signifikan atas Face murni pada SVM ($p = 0.0003$).
2. **Keadilan Algoritmik (Equal Opportunity)**:
   - Nilai *Equal Opportunity Difference* ($\Delta \text{TPR}$) pada Tri-SVM tercatat sebesar **7.50%** (White Males 96.94% vs Black Females 89.44%).
3. **Determinisme dan Reproduktibilitas**:
   - Seluruh 28 eksperimen bertumpu pada satu partisi 80/20 terkunci (`random_state=42`) dengan 8,640 sampel latih dan 2,160 sampel uji held-out.
